# Particle Tracking — Main Notebook (Data, Training, Inference)



This notebook is the main entry point and uses the utilities in `src/` to:

- Build synthetic datasets with ground truth (`src/build_datasets.py`)

- Train the detector (heatmap U-Net) and the Z regressor (`src/train.py`)

- Load trained models and launch an interactive inference viewer (`src/infer_utils.py`)



Quick guide:

- Optional: Run Cells 2–4 to generate data and train models (can be time-consuming).

- Then run Cells 5–6 to load models and explore predictions interactively.



For command-line workflows, prefer running the scripts in `src/` directly.

In [ ]:
# Global imports and configuration (keeps all original capabilities)

# Base deps
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras

# Project modules
from src.utilities import seed_everything
from src.build_datasets import build_datasets_with_gt
from src.train import train_detector, train_z_regressor
from src.infer_utils import (
    build_infer_functions,
    make_browse,
    launch_interact,
)

# Runtime configuration (adjust as needed)
CONFIG = {
    # Data generation
    "generator_path": "src/Particle_Tracking_Training_Data.py",
    "out_dir": "./particles_data",
    "Nt": 1,
    "n_train": 2000,
    "n_val": 300,
    "n_test": 300,
    "heat_sigma": 2.0,
    # Training
    "patch_size": 21,
    "seed": 42,
    "det_epochs": 20,
    "det_batch": 8,
    "det_lr": 1e-3,
    "z_epochs": 20,
    "z_batch": 64,
    "z_lr": 1e-3,
}

# Set global seeds (NumPy + TensorFlow)
seed_everything(CONFIG["seed"])

In [17]:
# — Optional Step 1: Build synthetic datasets with GT (train/val/test) —
# Note: This can take time. Run once, the results are saved under particles_data/ by default.



# build_datasets_with_gt(
#     generator_path=CONFIG["generator_path"],
#     out_dir=CONFIG["out_dir"],
#     n_train=CONFIG["n_train"], n_val=CONFIG["n_val"], n_test=CONFIG["n_test"],
#     Nt=CONFIG["Nt"], heat_sigma=CONFIG["heat_sigma"], seed=CONFIG["seed"],
# )

In [5]:
# — Optional Step 2: Train detector (U-Net heatmap). Saves detector.keras under the data folder. —



# tr_npz_default = os.path.join(CONFIG["out_dir"], "train.npz")
# va_npz_default = os.path.join(CONFIG["out_dir"], "val.npz")
# tr_npz_wgt = os.path.join(CONFIG["out_dir"], "train_with_gt.npz")
# va_npz_wgt = os.path.join(CONFIG["out_dir"], "val_with_gt.npz")
# tr_npz = tr_npz_wgt if os.path.exists(tr_npz_wgt) else tr_npz_default
# va_npz = va_npz_wgt if os.path.exists(va_npz_wgt) else va_npz_default
# det_model, _ = train_detector(
#     tr_npz, va_npz,
#     epochs=CONFIG["det_epochs"], batch_size=CONFIG["det_batch"], lr=CONFIG["det_lr"]
# )
# det_model.save("models/detector_test.keras")

In [4]:
# — Optional Step 3: Train Z regressor (patch-based). Saves z_regressor.keras under the data folder. —



# z_model, _ = train_z_regressor(
#     tr_npz, va_npz,
#     patch_size=CONFIG["patch_size"],
#     epochs=CONFIG["z_epochs"], batch_size=CONFIG["z_batch"], lr=CONFIG["z_lr"]
# )
# z_model.save("models/z_regressor_test.keras")

## Predictions Results

In [2]:
# Load trained models (from models/ by default). Skip if already in memory.

# To load models produced by training above, switch to CONFIG["out_dir"] paths:
# detector_path = os.path.join(CONFIG["out_dir"], "detector.keras")
# zreg_path     = os.path.join(CONFIG["out_dir"], "z_regressor.keras")

detector_path = os.path.join("models/detector.keras")
zreg_path     = os.path.join("models/z_regressor.keras")

# Idempotent loading: only load if the variable is not already defined
try:
    det
except NameError:
    det = keras.models.load_model(detector_path, compile=False)

try:
    znet
except NameError:
    znet = keras.models.load_model(zreg_path, compile=False)

In [ ]:
# Inference and interactive viewer (via src/infer_utils)
# - Load test split with GT from particles_data/test_with_gt.npz
# - Build compiled det_step/z_step functions from loaded models
# - Widget parameters:
#   i      -> frame index
#   thr    -> NMS threshold on heatmap
#   px_tol -> pixel tolerance for GT matching
#   patch  -> patch size for z-regressor input (auto-resized to z model's training size)

# Data path
data_dir = "./particles_data"
test_npz = os.path.join(data_dir, "test_with_gt.npz")

# Build inference functions (reuses det/znet from previous cell)
det_step, z_step, Z_PATCH = build_infer_functions(det, znet)

# Load test data (allow_pickle=True due to variable-length GT lists)
te = np.load(test_npz, allow_pickle=True)
imgs = te["imgs"]               # (N,H,W,1)
label_heatmaps = te["heatmaps"] # (N,H,W,1)
gt_xy_list = te["gt_xy_list"]   # object array per image
gt_z_list  = te["gt_z_list"]    # object array per image

# Launch interactive viewer
browse = make_browse(imgs, label_heatmaps, gt_xy_list, gt_z_list, det_step, z_step, Z_PATCH)
N = imgs.shape[0]
launch_interact(browse, N)

interactive(children=(IntSlider(value=0, description='index', max=29), FloatSlider(value=0.3, description='pre…

<function src.infer_utils.make_browse.<locals>._browse(i: 'int' = 0, thr: 'float' = 0.3, px_tol: 'float' = 3.0, patch_size: 'int' = 21, show_label_heat: 'bool' = True, show_pred_z: 'bool' = True, show_gt_z: 'bool' = True)>